In [25]:
# Run this cell to install the required packages
!pip install google-adk requests geopy

In [26]:
import os

os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = "qwiklabs-gcp-04-23b9cb055c5f"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"


Tool to find location and check the weather for that location from NWS

In [27]:
import requests
from geopy.geocoders import Nominatim

def get_nws_weather(city: str) -> str:
    """
    Retrieves the 7-day weather forecast for a specified US city using the National Weather Service API.

    Args:
        city (str): The name of the US city (e.g., "Seattle, WA", "New York, NY").

    Returns:
        str: A detailed description of the weather forecast for the upcoming week.
    """
    try:
        # 1. Convert the city name into coordinates
        geolocator = Nominatim(user_agent="adk_weather_lab")
        location = geolocator.geocode(city)

        if not location:
            return f"Error: Could not find coordinates for {city}."

        lat, lon = round(location.latitude, 4), round(location.longitude, 4)

        # 2. Call NWS to get the grid forecast URL
        headers = {"User-Agent": "(adk-student-lab, student@example.com)"}
        points_url = f"https://api.weather.gov/points/{lat},{lon}"
        points_resp = requests.get(points_url, headers=headers).json()

        if "properties" not in points_resp:
            return "Error: Location may be outside the US. NWS only covers US territories."

        forecast_url = points_resp["properties"]["forecast"]

        # 3. Call the forecast URL and extract the 7-day period array
        forecast_resp = requests.get(forecast_url, headers=headers).json()
        periods = forecast_resp["properties"]["periods"]

        # Compile the full week's forecast (up to 14 day/night periods) to pass back to the LLM
        forecast_summary = [f"Forecast for {city}:"]
        for period in periods:
            forecast_summary.append(f"- {period['name']}: {period['detailedForecast']}")

        return "\n".join(forecast_summary)

    except Exception as e:
        return f"Error fetching weather: {str(e)}"

Agent Setup

In [28]:
from google.adk.agents import Agent

weather_agent = Agent(
    name="weather_bot",
    model="gemini-2.5-flash",
    description="An agent that provides accurate weather forecasts.",
    instruction="""
    You are a helpful meteorology assistant.
    Always use the get_nws_weather tool when a user asks about the weather.

    CRITICAL FORMATTING RULES:
    1. The weather tool returns a full 7-day forecast.
    2. DEFAULT RULE: If the user DOES NOT specify a day or time (e.g., "What is the weather in Seattle?"), you MUST ONLY provide the forecast for the current day and night.
    3. SPECIFIC DAY RULE: If the user specifies a day (e.g., "What is the weather on Wednesday?"), extract and provide ONLY the forecast for that specific day.
    4. Never dump the entire 7-day forecast unless the user explicitly asks for "the week".
    """,
    tools=[get_nws_weather]
)

print(f"Agent '{weather_agent.name}' is updated and ready!")

Agent 'weather_bot' is updated and ready!


Test Agent

In [29]:
import asyncio
from google.adk.runners import InMemoryRunner
from google.genai import types

# 1. Initialize the runner for your notebook
runner = InMemoryRunner(
    agent=weather_agent,
    app_name="weather_lab"
)

# 2. Define IDs to keep track of conversation memory across turns
session_id = "lab_session_1"
user_id = "student"

# NEW: Explicitly create the session in memory before we start chatting
await runner.session_service.create_session(
    app_name="weather_lab",
    user_id=user_id,
    session_id=session_id
)

print("Weather Bot: Hello! What US city would you like the forecast for? (type 'quit' to exit)")

# 3. Use an async function since ADK runners stream events asynchronously
async def chat_loop():
    while True:
        user_input = input("You: ")
        if user_input.lower() in ['quit', 'exit']:
            break

        # Package the user's string into the Content format ADK expects
        message_content = types.Content(
            role="user",
            parts=[types.Part.from_text(text=user_input)]
        )

        # Run the agent and iterate through its execution events
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session_id,
            new_message=message_content
        ):
            # Only print when the agent has finished its tool calls and formulated a final answer
            if event.is_final_response():
                if event.content and event.content.parts:
                    print(f"Weather Bot: {event.content.parts[0].text}")

# Start the interactive loop
await chat_loop()

Weather Bot: Hello! What US city would you like the forecast for? (type 'quit' to exit)
You: what is the weather in Haymarket, va
Weather Bot: The weather in Haymarket, VA, this afternoon will be sunny, with a high near 70 and a northwest wind around 5 mph. Tonight, it will be mostly clear, with a low around 45 and a northwest wind around 3 mph.
You: will it rain at all this week
Weather Bot: Yes, there is a slight chance of rain showers on Sunday and Sunday Night.
You: how about in San Diego
Weather Bot: In San Diego, it will be sunny today with a high near 79, and a northeast wind of 0 to 10 mph with gusts up to 20 mph. Tonight, there will be patchy fog after 11 pm, partly cloudy with a low around 60, and a northeast wind of 5 to 10 mph.

Regarding rain this week in San Diego:
*   There is a slight chance of rain showers after 11 pm on Thursday night.
*   There's a slight chance of rain showers on Friday.
*   There's a chance of rain showers on Friday night.
*   There's a chance of r